# ĐỀ TÀI: MÔ HÌNH GỢI Ý CONCEPT THIẾT KẾ NỘI THẤT DỰA TRÊN NHU CẦU NGƯỜI DÙNG

### Data source: https://www.kaggle.com/datasets/zara2099/interior-design-material-and-style-dataset?select=interior_design_dataset.csv

# 1.Cài đặt thư viện

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder
import re
import joblib

# 2. Tải Data

In [2]:
df = pd.read_csv('Interior Design Material & Style Dataset/interior_design_dataset.csv')
print(df.head(10))

   Image_id  material_image_id    style_tags  \
0         1                  1    industrial   
1         2                  2    minimalist   
2         3                  3    industrial   
3         4                  4          boho   
4         5                  5    minimalist   
5         6                  6    industrial   
6         7                  7  scandinavian   
7         8                  8  scandinavian   
8         9                  9    industrial   
9        10                 10          boho   

                                  design_description  \
0  A modern loft with exposed brick and steel fin...   
1  A simple, clean space with neutral colors and ...   
2  A warehouse-inspired kitchen featuring raw mat...   
3  Bohemian style living room full of plants and ...   
4  A bedroom with white walls, simple furniture, ...   
5  A modern loft with exposed brick and steel fin...   
6  Scandinavian style kitchen emphasizing light a...   
7  A cozy bedroom with 

# 3. Tiền xử lí dữ liệu

lowcase text

In [3]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df['clean_design_description'] = df['design_description'].apply(clean_text)

chia cột màu thành 3 cột màu riêng biệt

In [4]:
df[['color1', 'color2', 'color3']] = df['color_palette'].str.split(',', expand=True)

Xử lý Nhãn Đơn

In [5]:
# 1. Khởi tạo encoder
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# 2. Fit và transform cả 2 cột cùng một lúc
columns_to_encode = ['style_tags', 'material_type']
encoded_matrix = encoder.fit_transform(df[columns_to_encode])

# 3. Trích xuất tên cột mới cho tất cả các cột đã encode
new_column_names = encoder.get_feature_names_out(columns_to_encode)

# 4. Đưa ma trận số vừa tạo thành một DataFrame mới
df_encoded = pd.DataFrame(encoded_matrix, columns=new_column_names)

# 5. Nối vào bảng gốc (nhớ dùng join hoặc concat đảm bảo index khớp nhau)
df_final = pd.concat([df, df_encoded], axis=1)

xóa cột cũ

In [6]:
df_final.drop(columns='style_tags', inplace=True)
df_final.drop(columns='material_type', inplace=True)
df_final.drop(columns='design_description', inplace=True)
df_final.drop(columns='color_palette', inplace=True)

Sắp xếp lại thứ tự các c

In [7]:
cot_moi = ['Image_id','material_image_id','style_tags_boho','style_tags_industrial','style_tags_minimalist','style_tags_scandinavian','material_type_fabric','material_type_glass','material_type_leather','material_type_marble','material_type_metal','material_type_stone','material_type_tile','material_type_wood','clean_design_description','user_preference','color1','color2','color3']
df_final = df_final[cot_moi]

## DŨ LIỆU SAU KHI TIỀN XỬ LÝ

In [8]:
print(df_final.head(10))

   Image_id  material_image_id  style_tags_boho  style_tags_industrial  \
0         1                  1              0.0                    1.0   
1         2                  2              0.0                    0.0   
2         3                  3              0.0                    1.0   
3         4                  4              1.0                    0.0   
4         5                  5              0.0                    0.0   
5         6                  6              0.0                    1.0   
6         7                  7              0.0                    0.0   
7         8                  8              0.0                    0.0   
8         9                  9              0.0                    1.0   
9        10                 10              1.0                    0.0   

   style_tags_minimalist  style_tags_scandinavian  material_type_fabric  \
0                    0.0                      0.0                   0.0   
1                    1.0           

Lưu bộ mã hóa để bàn giao cho Huyền (Backend)

In [9]:
joblib.dump(encoder, 'style_onehot_encoder.pkl')

['style_onehot_encoder.pkl']

Lưu file dữ liệu dạng số để giao cho Phương Anh (ML Model)

In [10]:
df_final.to_csv('dataset_cleaned_ready.csv', index=False)